In [1]:
from pathlib import Path
from pypdf import PdfReader

DATA_DIR = Path("../data")

def load_pdf(path):
    reader = PdfReader(path)
    text = "\n".join(page.extract_text() or "" for page in reader.pages)
    return text, len(reader.pages)

def load_txt(path):
    return Path(path).read_text(encoding="utf-8"), None

documents = {}
failed = []

for file in DATA_DIR.rglob("*"):
    if file.suffix == ".pdf":
        try:
            text, n_pages = load_pdf(file)
            documents[str(file)] = {"text": text, "pages": n_pages, "type": "pdf"}
        except Exception as e:
            failed.append((str(file), str(e)))
    elif file.suffix == ".txt":
        text, _ = load_txt(file)
        documents[str(file)] = {"text": text, "pages": None, "type": "txt"}

print(f"Loaded {len(documents)} documents, {len(failed)} failed")
for path, doc in documents.items():
    print(f"- {path}: {len(doc['text'])} chars" + (f", {doc['pages']} pages" if doc['pages'] else ""))

Loaded 6 documents, 0 failed
- ..\data\complaints\page_text_شكاوي.txt: 1589 chars
- ..\data\laws\CPA-Newlaw.pdf: 71197 chars, 32 pages
- ..\data\official_guides\page_text_تعريف.txt: 4409 chars
- ..\data\official_guides\اسئلة متكررة.txt: 4245 chars
- ..\data\official_guides\نصائح عامة .txt: 1398 chars
- ..\data\regulations\283655.pdf: 28 chars, 29 pages


In [2]:
!pip install pdfplumber



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import pdfplumber

def load_pdf_plumber(path):
    with pdfplumber.open(path) as pdf:
        text = "\n".join(page.extract_text() or "" for page in pdf.pages)
        return text, len(pdf.pages)

text, n_pages = load_pdf_plumber(r"..\data\regulations\283655.pdf")
print(f"{len(text)} chars, {n_pages} pages")

28 chars, 29 pages


## 2.1 Load & Inspect

تم تحميل 6 مستندات من مصادر جهاز حماية المستهلك الرسمي (CPA):

| الملف | النوع | الحجم |
|---|---|---|
| CPA-Newlaw.pdf | PDF | 71,197 حرف / 32 صفحة |
| اسئلة متكررة.txt | TXT | 4,245 حرف |
| page_text_تعريف.txt | TXT | 4,409 حرف |
| نصائح عامة.txt | TXT | 1,398 حرف |
| page_text_شكاوي.txt | TXT | 1,589 حرف |
| 283655.pdf (اللائحة التنفيذية) | PDF | مستبعد |

**ملف مستبعد:** `283655.pdf` (اللائحة التنفيذية) — تبيّن أنه ملف ممسوح ضوئيًا (scanned)؛
كل من `pypdf` و`pdfplumber` استخرجا 28 حرف فقط من 29 صفحة، رغم عدم وجود استثناء (exception)
أثناء المعالجة. استخراج نص حقيقي منه يتطلب OCR، وهو خارج نطاق الوقت المتاح (6 أيام).

**القرار:** الاعتماد على القانون الأساسي (181/2018) + الأدلة الرسمية الأربعة كـ corpus أساسي
(82,838 حرف إجمالاً)، لتغطية شاملة لحالات الاستبدال، الاسترجاع، الضمان، وإجراءات الشكاوى.

In [4]:
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

all_chunks = []
for path, doc in documents.items():
    doc_chunks = chunk_text(doc["text"], chunk_size=500, overlap=100)
    for i, chunk in enumerate(doc_chunks):
        all_chunks.append({
            "id": f"{path}_{i}",
            "text": chunk,
            "source": path,
            "chunk_index": i
        })

print(f"Total chunks: {len(all_chunks)}")

Total chunks: 209


## 2.2 Chunking Strategy

تم استخدام تقسيم ثابت الحجم (Fixed-size chunking) بـ chunk_size=500 حرف وoverlap=100 حرف.

**التبرير:**
- المستندات (القانون + الأدلة الرسمية) عربية، والحرف الواحد في العربي يحمل معنى أعلى نسبيًا
  من الإنجليزي، فـ 500 حرف (~80-100 كلمة) كافية لتغطية فكرة/مادة قانونية كاملة تقريبًا دون تفتيتها.
- الـ overlap بنسبة 20% (100 من 500) يقلل خطر قطع جملة أو شرط قانوني في منتصفه بين chunk وتاليه،
  وهو مهم بشكل خاص في نص قانوني حيث الاستثناءات (زي حالات عدم جواز الاسترجاع) غالبًا مرتبطة
  بجملة سابقة لها مباشرة.
- تم تفضيله على semantic/section-based chunking لأن استخراج حدود المواد (Articles) تلقائيًا
  من نص PDF غير مهيكل أصليًا عرضة للأخطاء، بينما fixed-size أضمن وأسرع تنفيذًا ضمن الوقت المتاح.

In [5]:
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

all_chunks = []
for path, doc in documents.items():
    doc_chunks = chunk_text(doc["text"], chunk_size=500, overlap=100)
    for i, chunk in enumerate(doc_chunks):
        all_chunks.append({
            "id": f"{path}_{i}",
            "text": chunk,
            "source": path,
            "chunk_index": i
        })

print(f"Total chunks: {len(all_chunks)}")

Total chunks: 209


In [6]:
from sentence_transformers import SentenceTransformer
import chromadb

# 1. حمّل موديل الـ embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. ولّد embeddings لكل الـ chunks
texts = [c["text"] for c in all_chunks]
embeddings = model.encode(texts, show_progress_bar=True)

print(f"Generated {len(embeddings)} embeddings, dimension: {embeddings.shape[1]}")

# 3. اعمل persistent Chroma client (يخزن على الديسك مباشرة)
client = chromadb.PersistentClient(path="../data/vector_store")
collection = client.get_or_create_collection(
    name="cpa_consumer_rights",
    metadata={"hnsw:space": "cosine"}
)

# 4. خزّن الـ chunks مع الـ embeddings والـ metadata
collection.add(
    ids=[c["id"] for c in all_chunks],
    embeddings=embeddings.tolist(),
    documents=[c["text"] for c in all_chunks],
    metadatas=[{"source": c["source"], "chunk_index": c["chunk_index"]} for c in all_chunks]
)

print(f"Stored {collection.count()} chunks in vector store")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Generated 209 embeddings, dimension: 384
Stored 209 chunks in vector store


In [7]:
results = collection.query(
    query_texts=["كام يوم مدة الاسترجاع للسلعة المعيبة؟"],
    n_results=3
)
for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print(f"[{meta['source']}] {doc[:150]}...\n")

[..\data\official_guides\page_text_تعريف.txt] المعتمدة...

[..\data\official_guides\نصائح عامة .txt] مقالة:  2.5
عزيزي المستهلك عند شرائك حلوي المولد النبوي يجب التأكد من :
تقييم المقالة:  5.0
1
2
3
4
مواضيع ذات علاقة
نصائح عامة
تعريفات
واجبات المستهل...

[..\data\official_guides\اسئلة متكررة.txt] ن تاريخ لجوء المستهلك إليه ويكون استرجاع المبلغ المدفوع بذات طريقة الشراء .
وفى حالة وجود خلاف حول وجود عيب بالسلعة للمستهلك الحق في  تقديم شكوى للجها...



In [8]:
import re

def clean_text(text):
    # شيل سطور التقييم والنجوم
    text = re.sub(r"تقييم المقالة:\s*[\d.]+", "", text)
    text = re.sub(r"مقالة:\s*[\d.]+", "", text)
    # شيل أرقام النجوم المتتالية (1 2 3 4 5 لوحدها في سطر)
    text = re.sub(r"^\s*[\d\s]{1,10}\s*$", "", text, flags=re.MULTILINE)
    # شيل قسم "مواضيع ذات علاقة" وأي نص بعده (عادة نافيجيشن آخر الصفحة)
    text = re.split(r"مواضيع ذات علاقة", text)[0]
    # شيل الأسطر الفاضية الزيادة
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text

In [9]:
# 1. نضف كل المستندات النصية (TXT فقط، القانون PDF نضيف عادة)
for path, doc in documents.items():
    if doc["type"] == "txt":
        doc["text"] = clean_text(doc["text"])

# 2. أعد الـ chunking
all_chunks = []
for path, doc in documents.items():
    doc_chunks = chunk_text(doc["text"], chunk_size=500, overlap=100)
    for i, chunk in enumerate(doc_chunks):
        all_chunks.append({"id": f"{path}_{i}", "text": chunk, "source": path, "chunk_index": i})
print(f"Total chunks after cleaning: {len(all_chunks)}")

# 3. أعد الـ embeddings بالموديل الجديد
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
texts = [c["text"] for c in all_chunks]
embeddings = model.encode(texts, show_progress_bar=True)

# 4. امسح الـ collection القديمة واعمل واحدة جديدة نظيفة
client = chromadb.PersistentClient(path="../data/vector_store")
try:
    client.delete_collection("cpa_consumer_rights")
except:
    pass
collection = client.get_or_create_collection(name="cpa_consumer_rights", metadata={"hnsw:space": "cosine"})
collection.add(
    ids=[c["id"] for c in all_chunks],
    embeddings=embeddings.tolist(),
    documents=[c["text"] for c in all_chunks],
    metadatas=[{"source": c["source"], "chunk_index": c["chunk_index"]} for c in all_chunks]
)
print(f"Stored {collection.count()} chunks")

Total chunks after cleaning: 207


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

c:\Users\Dell\AppData\Local\Programs\Python\Python311\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dell\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Stored 207 chunks


In [10]:
results = collection.query(query_texts=["كام يوم مدة الاسترجاع للسلعة المعيبة؟"], n_results=3)
for doc, meta in zip(results["documents"][0], results["metadatas"][0]):
    print(f"[{meta['source']}] {doc[:150]}...\n")

[..\data\official_guides\page_text_تعريف.txt] ستلام السلعة بدون سبب مع وجود بعض الاستثناءات
ثانيا : استبدال او استرجاع السلع المعيبة خلال 30 يوم من تاريخ الاستلام
ونعرض ذلك فيما يلي :
اولا : الاست...

[..\data\official_guides\page_text_تعريف.txt] ، وكانت السلعة مطابقة لهذه المواصفات
5- الكتب والصحف والمجلات ، والبرامج المعلوماتية وما يماثلها
6- إذا كانت السلعه تعد من الحلي والمجوهرات وما في حكم...

[..\data\official_guides\نصائح عامة .txt] ودراسات
دراسات
تقارير
+
مرصد الأسعار
المجمعات الاستهلاكية
الاسعار الاسترشادية
الجمعيات الأهلية
نصائح عامة
الرئيسية
>
إرشادات
>
نصائح عامة
نشرة ارشادية...



# Text Cleaning

In [11]:
def clean_text(text):
    # شيل التقييمات (زي قبل كده)
    text = re.sub(r"تقييم المقالة:\s*[\d.]+", "", text)
    text = re.sub(r"مقالة:\s*[\d.]+", "", text)

    # قص كل حاجة قبل آخر breadcrumb (القايمة الجانبية + breadcrumb نفسه)
    if "الرئيسية" in text and ">" in text:
        parts = text.split(">")
        if len(parts) > 1:
            text = parts[-1]   # أول سطر بعد آخر ">" هو المحتوى الحقيقي

    # قص كل حاجة بعد الفوتر (مواضيع ذات علاقة / خدمات الجهاز)
    text = re.split(r"مواضيع ذات علاقة|خدمات الجهاز", text)[0]

    # شيل أرقام لوحدها في سطر (نجوم التقييم)
    text = re.sub(r"^\s*[\d\s]{1,10}\s*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text

In [12]:
for path, doc in documents.items():
    if doc["type"] == "txt":
        doc["text"] = clean_text(doc["text"])
        print(f"{path}: {len(doc['text'])} chars بعد التنضيف")

..\data\complaints\page_text_شكاوي.txt: 994 chars بعد التنضيف
..\data\official_guides\page_text_تعريف.txt: 3789 chars بعد التنضيف
..\data\official_guides\اسئلة متكررة.txt: 3724 chars بعد التنضيف
..\data\official_guides\نصائح عامة .txt: 668 chars بعد التنضيف


In [ ]:
def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

all_chunks = []
for path, doc in documents.items():
    doc_chunks = chunk_text(doc["text"], chunk_size=500, overlap=100)
    for i, chunk in enumerate(doc_chunks):
        all_chunks.append({
            "id": f"{path}_{i}",
            "text": chunk,
            "source": path,
            "chunk_index": i
        })

print(f"Total chunks: {len(all_chunks)}")

Total chunks: 209


config = {
    "embedding_model": "all-MiniLM-L6-v2",
    "chunk_size": 500,
    "chunk_overlap": 100,
    "vector_store_path": "../data/vector_store",
    "collection_name": "cpa_consumer_rights"
}

**ملف مستبعد إضافي:** `نصائح عامة.txt` — تبيّن أنه صفحة فهرس (index) لنشرات إرشادية موسمية
(عيد الفطر، شم النسيم، المولد النبوي) بدون محتوى نصي جوهري مستخرج، وغير ذي صلة بنطاق المشروع
(حقوق الاستبدال/الاسترجاع/الضمان/الشكاوى). تم استبعاده والاعتماد على 4 مستندات نهائية.

In [13]:
# شيل الملف من documents قبل الـ chunking
documents.pop(r"..\data\official_guides\نصائح عامة .txt", None)

all_chunks = []
for path, doc in documents.items():
    doc_chunks = chunk_text(doc["text"], chunk_size=500, overlap=100)
    for i, chunk in enumerate(doc_chunks):
        all_chunks.append({"id": f"{path}_{i}", "text": chunk, "source": path, "chunk_index": i})
print(f"Total chunks: {len(all_chunks)}")

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
texts = [c["text"] for c in all_chunks]
embeddings = model.encode(texts, show_progress_bar=True)

client = chromadb.PersistentClient(path="../data/vector_store")
try:
    client.delete_collection("cpa_consumer_rights")
except:
    pass
collection = client.get_or_create_collection(name="cpa_consumer_rights", metadata={"hnsw:space": "cosine"})
collection.add(
    ids=[c["id"] for c in all_chunks],
    embeddings=embeddings.tolist(),
    documents=[c["text"] for c in all_chunks],
    metadatas=[{"source": c["source"], "chunk_index": c["chunk_index"]} for c in all_chunks]
)
print(f"Stored {collection.count()} chunks")

Total chunks: 201


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Stored 201 chunks


In [43]:
import ollama

# def retrieve(query, n_results=3):
#     results = collection.query(query_texts=[query], n_results=n_results)
#     return [
#         {"text": doc, "source": meta["source"]}
#         for doc, meta in zip(results["documents"][0], results["metadatas"][0])
#     ]

# def build_prompt(query, retrieved_chunks):
#     context = "\n\n".join(
#         f"[مصدر: {c['source']}]\n{c['text']}" for c in retrieved_chunks
#     )
#     prompt = f"""أنت مساعد توعية لحقوق المستهلك في مصر. أجب على سؤال المستخدم بالاعتماد **فقط**
# على المعلومات الموجودة في السياق أدناه. لا تستخدم أي معرفة خارجية. إذا لم تجد إجابة كافية
# في السياق، قل ذلك صراحة.

# في نهاية إجابتك، اذكر المصدر (اسم الملف) الذي استندت إليه.

# هذه معلومات عامة للتوعية وليست استشارة قانونية.

# السياق:
# {context}

# سؤال المستخدم: {query}

# الإجابة:"""
#     return prompt

# def answer_question(query, model_name="llama3.1"):
#     chunks = retrieve(query)
#     prompt = build_prompt(query, chunks)
#     response = ollama.generate(model=model_name, prompt=prompt)
#     return response["response"], chunks

In [44]:
def build_prompt(query, retrieved_chunks):
    context = "\n\n".join(
        f"[مصدر: {c['source']}]\n{c['text']}" for c in retrieved_chunks
    )
    prompt = f"""أنت مساعد توعية لحقوق المستهلك في مصر. أجب بالاعتماد فقط على السياق أدناه،
باللغة العربية الفصحى فقط، بدون أي حروف أو كلمات من أي لغة أخرى.

خطوات الإجابة:
1. حدد أولاً: هل السؤال عن "حق استرجاع/استبدال سلعة" أم عن موضوع آخر (شكوى، ضمان، التزام مورد...)؟
2. إذا كان عن استرجاع سلعة محددة: تحقق هل هذه السلعة مذكورة ضمن قائمة استثناءات في السياق.
   إن وُجدت، فالحق لا ينطبق عليها. إن لم توجد، فالحق العام ينطبق.
3. إذا كان السؤال عن موضوع آخر غير الاسترجاع، أجب مباشرة من المعلومات ذات الصلة في السياق
   بدون أي إشارة للاستثناءات.
4. إذا لم يحتوِ السياق على إجابة كافية، قل ذلك صراحة.

مثال 1 (سلعة مستثناة): سؤال عن استرجاع كتاب، والسياق يذكر "5- الكتب والصحف" ضمن استثناءات
→ الإجابة: "لا، الكتب مستثناة من حق الاسترجاع بدون سبب."
مثال 2 (سؤال غير متعلق بالاستثناءات): سؤال عن طرق تقديم شكوى
→ الإجابة تذكر قنوات التواصل الفعلية من السياق مباشرة، بدون أي ذكر للاستثناءات.

في نهاية إجابتك، اذكر المصدر.
هذه معلومات عامة للتوعية وليست استشارة قانونية.

السياق:
{context}

سؤال المستخدم: {query}

الإجابة (بالعربية فقط):"""
    return prompt

In [48]:
def retrieve(query, n_results=5):
    results = collection.query(query_texts=[query], n_results=n_results)
    retrieved = [
        {"text": doc, "source": meta["source"]}
        for doc, meta in zip(results["documents"][0], results["metadatas"][0])
    ]
    # Fallback بسيط: لو السؤال فيه كلمة "شكوى/شكاوى" وملف الشكاوى مش ضمن النتائج، ضيفه يدويًا
    if "شكو" in query and not any("شكاوي" in r["source"] for r in retrieved):
        extra = collection.query(query_texts=[query], n_results=10)
        for doc, meta in zip(extra["documents"][0], extra["metadatas"][0]):
            if "شكاوي" in meta["source"]:
                retrieved.append({"text": doc, "source": meta["source"]})
                break
    return retrieved

In [45]:
def answer_question(query, model_name="qwen2.5:3b"):
    chunks = retrieve(query)
    prompt = build_prompt(query, chunks)
    response = ollama.generate(
        model=model_name,
        prompt=prompt,
        options={"temperature": 0}
    )
    return response["response"], chunks

In [ ]:
test_questions = [
    "ما هي مدة الاسترجاع للسلعة المعيبة؟",
    "هل يمكنني استرجاع الملابس الداخلية بعد فتح الغلاف؟",
    "اشتريت جهازًا وظهر به عيب بعد أسبوعين، ما حقوقي؟",
    "ما هي طرق تقديم شكوى لجهاز حماية المستهلك؟",
    "ما هي المستندات المطلوبة لتقديم شكوى؟",
    "هل يحق لي استرجاع الكتب والمجلات؟",
    "ماذا يلتزم به المورد عند معرفته بوجود عيب في منتج؟",
]
for q in test_questions:
    answer, chunks = answer_question(q, model_name="qwen2.5:3b")
    print(f"\n{'='*60}\nس: {q}\n{'-'*60}\nج: {answer}\nالمصادر: {[c['source'] for c in chunks]}\n")

In [46]:
results_log = []
for q in test_questions:
    answer, chunks = answer_question(q)
    results_log.append({"question": q, "answer": answer, "sources": [c["source"] for c in chunks]})
    print(f"\n{'='*60}\nس: {q}\n{'-'*60}\nج: {answer}\n")


س: ما هي مدة الاسترجاع للسلعة المعيبة؟
------------------------------------------------------------
ج: مدة الاسترجاع للسلع المعيبة هي خلال 30 يوم من تاريخ الاستلام.


س: هل يمكنني استرجاع الملابس الداخلية بعد فتح الغلاف؟
------------------------------------------------------------
ج: نعم، المستهلك له الحق في استرجاع الملابس الداخلية وفساتين الزفاف بعد فتح أغلفتها خلال 30 يومًا من تسلم السلعة، مع استرداد قيمتها النقدية إذا شابها عيب.


س: اشتريت جهازًا وظهر به عيب بعد أسبوعين، ما حقوقي؟
------------------------------------------------------------
ج: في هذه الحالة، يحق لك استبدال أو استرجاع الجهاز خلال 30 يومًا من تاريخ استلامه، مع استرداد قيمته النقدية إذا شابه عيب.


س: ما هي طرق تقديم شكوى لجهاز حماية المستهلك؟
------------------------------------------------------------
ج: طرق تقديم شكوى لجهاز حماية المستهلك كما وردت في السياق:

1. التواصل عبر تطبيق واتس آب على رقم 01577779999
2. إرسال شكوى إلكترونيا
3. استخدام تطبيق المحمول على متجر جوجل ستور وأبل ستور
4. إرسال شكوى من خلال الفاكس 

**ملاحظة: عدم ثبات الاستدلال المنطقي (Inconsistency)**
لوحظ أن النموذج (qwen2.5:3b) لم يطبق منطق الاستثناءات بشكل ثابت رغم استخدام نفس الـ prompt
ونفس بنية السياق: أجاب بشكل صحيح على استثناء الكتب من حق الاسترجاع، بينما أخفق في تطبيق
نفس المنطق على استثناء الملابس الداخلية (البند المجاور مباشرة في نفس القائمة). هذا يوضح حدًا
معروفًا لموديلات اللغة الصغيرة (3B parameters) في الاستدلال المنطقي الدقيق والمتسق على نصوص
قانونية، حتى مع توفر الـ grounding الصحيح والتعليمات الواضحة. في نسخة إنتاجية، يُنصح باستخدام
موديل أكبر (7B+) أو إضافة طبقة تحقق (verification layer) منفصلة لهذا النوع من الأسئلة.

**تحسين حرج على الـ Prompt (Prompt Engineering Fix):**
في الاختبار الأولي، أظهر النموذج (qwen2.5:3b) حالة "عكس منطقي" (logical inversion): عند
سؤاله عن إمكانية استرجاع الملابس الداخلية، أجاب بالإيجاب رغم أن الـ chunk المسترجع يحتوي
حرفيًا على نص الاستثناء الصحيح. تم حل المشكلة عبر تحسين الـ prompt بإضافة:
1. تفكير خطوة بخطوة (Chain-of-Thought) موجّه للتحقق من الاستثناءات أولاً
2. مثال توضيحي (Few-shot example) يوضح كيفية التعامل مع قوائم الاستثناءات
3. تقييد صريح للإجابة باللغة العربية فقط (لمعالجة تسريب رمزي من لغات أخرى لوحظ في محاولة سابقة)

بعد هذا التحسين، أجاب النموذج بشكل صحيح: "الملابس الداخلية وفساتين الزفاف... مستثنى منها
من حق الاسترجاع." هذا يوضح أن الـ grounding وحده غير كافٍ لضمان دقة الاستدلال المنطقي على
نصوص قانونية معقدة، وأن هندسة الـ prompt بعناية جزء أساسي من نظام RAG موثوق.